# Comprehensive Model Comparison: CNN vs LSTM vs LMU

This notebook compares the performance of:
- **CNN-only**: Spatial feature extraction
- **LSTM-only**: Temporal sequence modeling
- **LMU-only**: Legendre Memory Unit temporal encoding
- **CNN-LSTM (FallNet)**: Full ensemble baseline

**Goal**: Identify which architecture excels at which tasks to inform SNN conversion strategy.

In [ ]:
import numpy as np
import json
from pathlib import Path
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Configure matplotlib
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Class names
CLASS_NAMES = [
    'Walking',
    'Jogging', 
    'Walking_stairs_updown',
    'Stumble_while_walking',
    'Fall_Initiation',
    'Impact_Aftermath'
]

## 1. Setup Paths

In [ ]:
# Update these paths if needed
base_dir = Path.home() / 'repos/summerschool2023/projects/fall-detection/fall_detection_data'
model_dir = base_dir / 'models'
processed_dir = base_dir / 'processed'

print(f"Base directory: {base_dir}")
print(f"Model directory: {model_dir}")
print(f"Model directory exists: {model_dir.exists()}")
print(f"Processed directory exists: {processed_dir.exists()}")

## 2. Load Data

In [ ]:
# Load preprocessed data
X = np.load(base_dir / 'processed' / 'X_data_6class.npy')
y = np.load(base_dir / 'processed' / 'y_labels_6class.npy')

print(f"X shape: {X.shape}")
print(f"y shape (before): {y.shape}")

# Convert to one-hot encoding (models expect this format)
from tensorflow.keras.utils import to_categorical
y = to_categorical(y, num_classes=6)

print(f"y shape (after one-hot): {y.shape}")
print(f"Number of samples: {X.shape[0]}")
print(f"Sequence length: {X.shape[1]}")
print(f"Number of features: {X.shape[2]}")
print(f"Number of classes: {y.shape[1]}")

## 3. Define Evaluation Functions

In [ ]:
def evaluate_model(model, X, y, model_name):
    """Evaluate a single model and return metrics"""
    print(f"Evaluating {model_name}...")
    
    # Get predictions
    y_pred_probs = model.predict(X, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y, axis=1)
    
    # Overall metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Per-class metrics
    report = classification_report(y_true, y_pred, 
                                   target_names=CLASS_NAMES,
                                   output_dict=True,
                                   zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    print(f"  ✓ Accuracy: {accuracy:.4f}")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'per_class': report,
        'confusion_matrix': cm,
        'y_true': y_true,
        'y_pred': y_pred
    }



## 4. Load and Evaluate All Models

In [ ]:
# Model types to compare
import keras
import keras_lmu
import gc

model_types = {
    'CNN-only': 'cnn_only_fold_{}.keras',
    'LSTM-only': 'lstm_only_fold_{}.keras', 
    'LMU-only': 'lmu_only_fold_{}.keras',
    'CNN-LSTM (FallNet)': 'fallnet_fold_{}.keras'
}


results = {name: [] for name in model_types.keys()}

# Evaluate each model type across all folds
for model_name, pattern in model_types.items():
    print(f"\n{'='*60}")
    print(f"EVALUATING {model_name}")
    print("="*60)
    
    for fold in range(1, 6):
        model_path = model_dir / pattern.format(fold)
        
        if not model_path.exists():
            print(f"⚠️  {model_path.name} not found, skipping...")
            continue
        
        try:
    # Load model
            model = tf.keras.models.load_model(model_path)
    
    # Evaluate
            metrics = evaluate_model(model, X, y, f"{model_name} Fold {fold}")
            results[model_name].append(metrics)
    
    # Clear memory after each model
            del model
            gc.collect()
            tf.keras.backend.clear_session()
    
        except Exception as e:
            print(f"❌ Error loading {model_path.name}: {e}")
            continue
print("\n✓ All models evaluated!")

def evaluate_model(model, X, y, model_name, batch_size=32):
    """Evaluate a single model and return metrics - using batches to avoid OOM"""
    print(f"Evaluating {model_name}...")
    
    # Get predictions in batches to avoid OOM
    y_pred_probs = model.predict(X, batch_size=batch_size, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y, axis=1)
    
    # Overall metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Per-class metrics
    report = classification_report(y_true, y_pred, 
                                   target_names=CLASS_NAMES,
                                   output_dict=True,
                                   zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    print(f"  ✓ Accuracy: {accuracy:.4f}")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'per_class': report,
        'confusion_matrix': cm,
        'y_true': y_true,
        'y_pred': y_pred
    }

## 5. Aggregate Results Across Folds

def aggregate_results(results):
    """Aggregate metrics across folds"""
    aggregated = {}
    
    for model_name, fold_results in results.items():
        if not fold_results:
            continue
            
        # Overall metrics
        accuracies = [r['accuracy'] for r in fold_results]
        precisions = [r['precision'] for r in fold_results]
        recalls = [r['recall'] for r in fold_results]
        f1s = [r['f1'] for r in fold_results]
        
        # Per-class metrics (average across folds)
        per_class_metrics = {}
        for class_name in CLASS_NAMES:
            class_precisions = [r['per_class'][class_name]['precision'] for r in fold_results]
            class_recalls = [r['per_class'][class_name]['recall'] for r in fold_results]
            class_f1s = [r['per_class'][class_name]['f1-score'] for r in fold_results]
            
            per_class_metrics[class_name] = {
                'precision': (np.mean(class_precisions), np.std(class_precisions)),
                'recall': (np.mean(class_recalls), np.std(class_recalls)),
                'f1': (np.mean(class_f1s), np.std(class_f1s))
            }
        
        aggregated[model_name] = {
            'accuracy': (np.mean(accuracies), np.std(accuracies)),
            'precision': (np.mean(precisions), np.std(precisions)),
            'recall': (np.mean(recalls), np.std(recalls)),
            'f1': (np.mean(f1s), np.std(f1s)),
            'per_class': per_class_metrics,
            'n_folds': len(fold_results)
        }
    
    return aggregated

aggregated = aggregate_results(results)
print("✓ Results aggregated!")

## 6. Overall Performance Comparison

In [ ]:
# Create DataFrame for easy viewing
summary_data = []
for model_name, metrics in aggregated.items():
    summary_data.append({
        'Model': model_name,
        'Accuracy': f"{metrics['accuracy'][0]:.4f} ± {metrics['accuracy'][1]:.4f}",
        'Precision': f"{metrics['precision'][0]:.4f} ± {metrics['precision'][1]:.4f}",
        'Recall': f"{metrics['recall'][0]:.4f} ± {metrics['recall'][1]:.4f}",
        'F1-Score': f"{metrics['f1'][0]:.4f} ± {metrics['f1'][1]:.4f}",
        'Folds': metrics['n_folds']
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Accuracy', ascending=False)

print("\n" + "="*80)
print("OVERALL PERFORMANCE COMPARISON")
print("="*80)
print(summary_df.to_string(index=False))

## 7. Visualize Overall Performance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

metrics = ['accuracy', 'precision', 'recall', 'f1']
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for idx, (metric, metric_name) in enumerate(zip(metrics, metric_names)):
    ax = axes[idx // 2, idx % 2]
    
    models = list(aggregated.keys())
    means = [aggregated[m][metric][0] for m in models]
    stds = [aggregated[m][metric][1] for m in models]
    
    bars = ax.bar(range(len(models)), means, yerr=stds, capsize=5, alpha=0.7)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, rotation=45, ha='right')
    ax.set_ylabel(metric_name)
    ax.set_title(f'{metric_name} Comparison')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1.0])
    
    # Add value labels
    for bar, mean, std in zip(bars, means, stds):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{mean:.3f}\n±{std:.3f}',
               ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(model_dir / 'overall_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Fall_Initiation Performance (Critical Metric)

In [ ]:
# Create DataFrame for Fall_Initiation class
fall_data = []
for model_name, metrics in aggregated.items():
    fall_metrics = metrics['per_class']['Fall_Initiation']
    fall_data.append({
        'Model': model_name,
        'Precision': f"{fall_metrics['precision'][0]:.4f} ± {fall_metrics['precision'][1]:.4f}",
        'Recall': f"{fall_metrics['recall'][0]:.4f} ± {fall_metrics['recall'][1]:.4f}",
        'F1-Score': f"{fall_metrics['f1'][0]:.4f} ± {fall_metrics['f1'][1]:.4f}"
    })

fall_df = pd.DataFrame(fall_data)

print("\n" + "="*80)
print("FALL_INITIATION CLASS PERFORMANCE (Critical for Safety)")
print("="*80)
print(fall_df.to_string(index=False))

In [ ]:
# Visualize Fall_Initiation performance
fig, ax = plt.subplots(figsize=(10, 6))

models = list(aggregated.keys())
fall_precisions = [aggregated[m]['per_class']['Fall_Initiation']['precision'][0] for m in models]
fall_recalls = [aggregated[m]['per_class']['Fall_Initiation']['recall'][0] for m in models]
fall_f1s = [aggregated[m]['per_class']['Fall_Initiation']['f1'][0] for m in models]

x = np.arange(len(models))
width = 0.25

ax.bar(x - width, fall_precisions, width, label='Precision', alpha=0.8)
ax.bar(x, fall_recalls, width, label='Recall', alpha=0.8)
ax.bar(x + width, fall_f1s, width, label='F1-Score', alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Fall_Initiation Class Performance (Critical for Safety)', 
             fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1.0])

plt.tight_layout()
plt.savefig(model_dir / 'fall_initiation_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Per-Class Recall Heatmap

In [ ]:
# Create recall matrix
models = list(aggregated.keys())
recall_matrix = np.zeros((len(CLASS_NAMES), len(models)))

for i, class_name in enumerate(CLASS_NAMES):
    for j, model_name in enumerate(models):
        recall_matrix[i, j] = aggregated[model_name]['per_class'][class_name]['recall'][0]

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(recall_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=models, yticklabels=CLASS_NAMES,
            cbar_kws={'label': 'Recall'}, ax=ax, vmin=0, vmax=1)

ax.set_title('Per-Class Recall Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(model_dir / 'per_class_recall_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Detailed Per-Class Breakdown

In [ ]:
# Print per-class recall for all models
print("\n" + "="*80)
print("PER-CLASS RECALL COMPARISON")
print("="*80)

for class_name in CLASS_NAMES:
    print(f"\n{class_name}:")
    print(f"{'Model':<25} {'Recall':<20}")
    print("-" * 45)
    
    for model_name in models:
        rec_mean, rec_std = aggregated[model_name]['per_class'][class_name]['recall']
        print(f"{model_name:<25} {rec_mean:.4f} ± {rec_std:.4f}")

## 11. Save Results to JSON

In [ ]:
# Convert to JSON-serializable format
json_data = {}
for model_name, metrics in aggregated.items():
    json_data[model_name] = {
        'overall': {
            'accuracy': {'mean': float(metrics['accuracy'][0]), 'std': float(metrics['accuracy'][1])},
            'precision': {'mean': float(metrics['precision'][0]), 'std': float(metrics['precision'][1])},
            'recall': {'mean': float(metrics['recall'][0]), 'std': float(metrics['recall'][1])},
            'f1': {'mean': float(metrics['f1'][0]), 'std': float(metrics['f1'][1])},
        },
        'per_class': {}
    }
    
    for class_name, class_metrics in metrics['per_class'].items():
        json_data[model_name]['per_class'][class_name] = {
            'precision': {'mean': float(class_metrics['precision'][0]), 'std': float(class_metrics['precision'][1])},
            'recall': {'mean': float(class_metrics['recall'][0]), 'std': float(class_metrics['recall'][1])},
            'f1': {'mean': float(class_metrics['f1'][0]), 'std': float(class_metrics['f1'][1])},
        }

output_file = model_dir / 'model_comparison_summary.json'
with open(output_file, 'w') as f:
    json.dump(json_data, f, indent=2)

print(f"✓ Saved JSON summary to {output_file}")

## 12. Key Insights & Next Steps

Based on the results above, identify:

1. **Best overall performer** - Which model has highest accuracy?
2. **Safety critical** - Which model has best Fall_Initiation recall?
3. **Architecture strengths**:
   - Does CNN excel at spatial patterns (Walking vs Jogging)?
   - Does LSTM capture temporal dynamics (Fall_Initiation)?
   - Where does LMU struggle?
4. **SNN conversion strategy**:
   - If LMU is competitive → CNN-LMU ensemble (easier conversion)
   - If LSTM dominates → CNN-LSTM (harder conversion, higher baseline)
   - CNN-only always viable (easiest conversion, 88.8% baseline)